In [1]:
import pandas as pd
from datetime import datetime

path = r"C:\Users\NehaMathew\Documents\FFA\input\Capesize_FFA.csv"

df = pd.read_csv(path, sep=None, engine="python")

# Parse ArchiveDate
df["ArchiveDate"] = pd.to_datetime(df["ArchiveDate"], errors="coerce")

# Latest 5TC snapshot
latest = df["ArchiveDate"].max()
latest_df = df[
    (df["ArchiveDate"] == latest) &
    (df["RouteIdentifier"].astype(str).str.startswith("5TC_C"))
].copy()

# Clean description a bit
latest_df["FFADescription"] = latest_df["FFADescription"].astype(str).str.strip()

print("Unique FFADescription for latest 5TC rows:")
print(sorted(latest_df["FFADescription"].unique()))
print()

# Inspect one example monthly and one quarterly row so we know formats
print(latest_df[["RouteIdentifier", "RouteAverage", "FFADescription"]]
      .sort_values("FFADescription")
      .head(20))

# --- helper functions that you may need to tweak after seeing the above print ---

def month_label(dt: datetime) -> str:
    # Your data: 'Dec 25', 'Jan 26', ...  (space instead of dash)
    return dt.strftime("%b %y")   # e.g. 'Dec 25'


def quarter_label(dt: datetime) -> str:
    """
    Return label for a quarter row exactly as in FFADescription.
    Most likely 'Q4 25', but change if needed after looking at the print.
    """
    q = (dt.month - 1) // 3 + 1
    return f"Q{q} {dt.strftime('%y')}"
    # If you see 'Q4-25' use:
    # return f"Q{q}-{dt.strftime('%y')}"
    # If you see 'Q4 2025' use:
    # return f"Q{q} {dt.year}"

# --- build next 12 months from 1 Dec 2025 ---
start = datetime(2026, 2, 1)
months = pd.date_range(start, periods=12, freq="MS")

rows = []
for dt in months:
    mon_lab = month_label(dt)
    q_lab = quarter_label(dt)

    # Monthly first
    row_m = latest_df[latest_df["FFADescription"] == mon_lab]

    if not row_m.empty:
        use_row = row_m.iloc[0]
        source = "Monthly"
    else:
        # Quarterly fallback
        row_q = latest_df[latest_df["FFADescription"] == q_lab]
        if not row_q.empty:
            use_row = row_q.iloc[0]
            source = "Quarterly"
        else:
            use_row = None
            source = None

    rows.append({
        "Month": dt.strftime("%Y-%m"),
        "FFA_Description_Used": None if use_row is None else use_row["FFADescription"],
        "C5TC_FFA": None if use_row is None else use_row["RouteAverage"],
        "Source": source
    })

# ... all your existing code that builds result_df ...

result_df = pd.DataFrame(rows)
print(result_df)

# save to CSV (no index column)
output_path = r"C:\Users\NehaMathew\Documents\Myproject\FFA NEXT 12 MONTHS\Capesize_FFA_12m_strip.csv"
result_df.to_csv(output_path, index=False)



C:\Users\NehaMathew\AppData\Local\Temp\ipykernel_4116\3145404447.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ArchiveDate"] = pd.to_datetime(df["ArchiveDate"], errors="coerce")


Unique FFADescription for latest 5TC rows:
['Aug 26', 'Cal 27', 'Cal 28', 'Cal 29', 'Cal 30', 'Cal 31', 'Cal 32', 'Cal 33', 'Dec 26', 'Jul 26', 'Jun 26', 'Nov 26', 'Oct 26', 'Q1 27', 'Q2 26', 'Q2 27', 'Q3 26', 'Q3 27', 'Q4 26', 'Q4 27', 'Sep 26']

       RouteIdentifier  RouteAverage FFADescription
115517   5TC_C182+2MON       33724.0         Aug 26
253507      5TC_C+2MON       30221.0         Aug 26
253519      5TC_C+1CAL       26496.0         Cal 27
115529   5TC_C182+1CAL       29999.0         Cal 27
253520      5TC_C+2CAL       23057.0         Cal 28
115530   5TC_C182+2CAL       26560.0         Cal 28
253521      5TC_C+3CAL       21914.0         Cal 29
115531   5TC_C182+3CAL       25417.0         Cal 29
253522      5TC_C+4CAL       20968.0         Cal 30
115532   5TC_C182+4CAL       24471.0         Cal 30
253523      5TC_C+5CAL       19989.0         Cal 31
115533   5TC_C182+5CAL       23492.0         Cal 31
253524      5TC_C+6CAL       19389.0         Cal 32
115534   5TC_C182+6CAL  

In [2]:
import pandas as pd
from datetime import datetime

# Paths

ffa_path = r"C:\Users\NehaMathew\Documents\Myproject\FFA NEXT 12 MONTHS\Capesize_FFA_12m_strip.csv"
spot_path = r"C:\Users\NehaMathew\Documents\Myproject\Daily spot Ratio\Spot_Ratio_Data_Week 24 2026.xlsx"
current_date_str = datetime.today().strftime("%Y-%m-%d")
baltic_path = fr"C:\Users\NehaMathew\Documents\Baltic\Baltic_Data_average_{current_date_str}.xlsx"

# Original ratio columns
RATIO_COLS_OLD = ["C8 VS C5TC", "C9 VS C5TC", "C10 VS C5TC", "C14 VS C5TC", "C16 VS C5TC"]

# New names with '_ratio' suffix
RATIO_COLS_NEW = [col.replace(" VS C5TC", " VS C5TC_ratio") for col in RATIO_COLS_OLD]

def build_block_for_years(year_list, label):
    ffa_df = pd.read_csv(ffa_path)
    ffa_df["Month_dt"] = pd.to_datetime(ffa_df["Month"] + "-01")
    ffa_df["MonthNum"] = ffa_df["Month_dt"].dt.month

    spot_df = pd.read_excel(spot_path, sheet_name="Capesize ratio data")
    spot_df["Date"] = pd.to_datetime(spot_df["Date"], errors="coerce")
    spot_df["Year"] = spot_df["Date"].dt.year
    spot_df["MonthNum"] = spot_df["Date"].dt.month

    spot_sel = spot_df[spot_df["Year"].isin(year_list)].copy()

    # Compute monthly averages for the old ratio columns
    monthly_avgs = (
        spot_sel.groupby("MonthNum", as_index=False)[RATIO_COLS_OLD].mean()
    )
    monthly_avgs[RATIO_COLS_OLD] = monthly_avgs[RATIO_COLS_OLD].round(2)

    # Rename old ratio columns to new names with '_ratio'
    monthly_avgs.rename(columns=dict(zip(RATIO_COLS_OLD, RATIO_COLS_NEW)), inplace=True)

    merged = ffa_df.merge(monthly_avgs, how="left", on="MonthNum")

    # Estimated FFA basis columns using new ratio names
    merged["C8_estimated_FFA basis"]  = merged["C5TC_FFA"] * merged["C8 VS C5TC_ratio"]
    merged["C9_estimated_FFA basis"]  = merged["C5TC_FFA"] * merged["C9 VS C5TC_ratio"]
    merged["C10_estimated_FFA basis"] = merged["C5TC_FFA"] * merged["C10 VS C5TC_ratio"]
    merged["C14_estimated_FFA basis"] = merged["C5TC_FFA"] * merged["C14 VS C5TC_ratio"]
    merged["C16_estimated_FFA basis"] = merged["C5TC_FFA"] * merged["C16 VS C5TC_ratio"]

    # Placeholder spot basis columns (will fill after stacking)
    for col in ["C8", "C9", "C10", "C14", "C16"]:
        merged[f"{col}_estimated_Spot basis"] = 0

    est_ffa_cols = [
        "C8_estimated_FFA basis", "C9_estimated_FFA basis",
        "C10_estimated_FFA basis", "C14_estimated_FFA basis", "C16_estimated_FFA basis"
    ]
    merged[est_ffa_cols] = merged[est_ffa_cols].round(0)

    merged = merged.drop(columns=["Month_dt", "MonthNum"])
    merged["Year"] = label

    cols = [
        "Month", "FFA_Description_Used", "C5TC_FFA", "Source",
        "C8 VS C5TC_ratio", "C8_estimated_FFA basis", "C8_estimated_Spot basis",
        "C9 VS C5TC_ratio", "C9_estimated_FFA basis", "C9_estimated_Spot basis",
        "C10 VS C5TC_ratio", "C10_estimated_FFA basis", "C10_estimated_Spot basis",
        "C14 VS C5TC_ratio", "C14_estimated_FFA basis", "C14_estimated_Spot basis",
        "C16 VS C5TC_ratio", "C16_estimated_FFA basis", "C16_estimated_Spot basis",
        "Year"
    ]

    return merged[cols]

# Read latest Baltic spot price
baltic_df = pd.read_excel(baltic_path, sheet_name="Daily")
baltic_df["Date"] = pd.to_datetime(baltic_df["Date"], errors="coerce")
latest_date = baltic_df["Date"].max()
latest_c5tc_spot = baltic_df.loc[baltic_df["Date"] == latest_date, "C5TC"].values[0]
date_str = latest_date.strftime("%d_%m_%Y")
spot_col_name = f"C5TC_{date_str}_Spot"

# Build year blocks 2018-2025
year_blocks = [build_block_for_years([y], y) for y in range(2018, 2027)]

# Last N years relative to base_year (2025)
base_year = 2025
last2_block = build_block_for_years([base_year - 1, base_year], "Last 2 years")
last3_block = build_block_for_years([base_year - 2, base_year - 1, base_year], "Last 3 years")
last4_block = build_block_for_years([base_year - 3, base_year - 2, base_year - 1, base_year], "Last 4 years")
last5_block = build_block_for_years([base_year - 4, base_year - 3, base_year - 2, base_year - 1, base_year], "Last 5 years")

# Stack all blocks
full_df = pd.concat(year_blocks + [last2_block, last3_block, last4_block, last5_block], ignore_index=True)

# Add spot column with constant value, insert after C5TC_FFA
full_df[spot_col_name] = latest_c5tc_spot
c5tc_idx = full_df.columns.get_loc("C5TC_FFA")
full_df = full_df.drop(columns=[spot_col_name])
full_df.insert(c5tc_idx + 1, spot_col_name, latest_c5tc_spot)

# Calculate estimated Spot basis columns with new ratio names
for col in ["C8", "C9", "C10", "C14", "C16"]:
    ratio_col = f"{col} VS C5TC_ratio"
    est_spot_col = f"{col}_estimated_Spot basis"
    full_df[est_spot_col] = (full_df[spot_col_name] * full_df[ratio_col]).round(0)

# Save full dataframe
output_path = r"C:\Users\NehaMathew\Downloads\Capesize_FFA_12m_full_with_last_ratios_and_spot.csv"
full_df.to_csv(output_path, index=False)

print(full_df.head(40))


      Month FFA_Description_Used  C5TC_FFA  C5TC_12_06_2026_Spot     Source  \
0   2026-02                  NaN       NaN               33748.0        NaN   
1   2026-03                  NaN       NaN               33748.0        NaN   
2   2026-04                Q2 26   39866.0               33748.0  Quarterly   
3   2026-05                Q2 26   39866.0               33748.0  Quarterly   
4   2026-06               Jun 26   38699.0               33748.0    Monthly   
5   2026-07               Jul 26   33114.0               33748.0    Monthly   
6   2026-08               Aug 26   33724.0               33748.0    Monthly   
7   2026-09               Sep 26   35507.0               33748.0    Monthly   
8   2026-10               Oct 26   35767.0               33748.0    Monthly   
9   2026-11               Nov 26   35439.0               33748.0    Monthly   
10  2026-12               Dec 26   34357.0               33748.0    Monthly   
11  2027-01                Q1 27   26210.0          

In [3]:
import pandas as pd
from datetime import datetime

path = r"C:\Users\NehaMathew\Documents\FFA\input\Handysize_FFA.csv"

df = pd.read_csv(path, sep=None, engine="python")

# Parse ArchiveDate
df["ArchiveDate"] = pd.to_datetime(df["ArchiveDate"], errors="coerce")

# Latest 5TC snapshot
latest = df["ArchiveDate"].max()
latest_df = df[
    (df["ArchiveDate"] == latest) &
    (df["RouteIdentifier"].astype(str).str.startswith("TC_H38"))
].copy()

# Clean description a bit
latest_df["FFADescription"] = latest_df["FFADescription"].astype(str).str.strip()

print("Unique FFADescription for latest 5TC rows:")
print(sorted(latest_df["FFADescription"].unique()))
print()

# Inspect one example monthly and one quarterly row so we know formats
print(latest_df[["RouteIdentifier", "RouteAverage", "FFADescription"]]
      .sort_values("FFADescription")
      .head(20))

# --- helper functions that you may need to tweak after seeing the above print ---

def month_label(dt: datetime) -> str:
    # Your data: 'Dec 25', 'Jan 26', ...  (space instead of dash)
    return dt.strftime("%b %y")   # e.g. 'Dec 25'


def quarter_label(dt: datetime) -> str:
    """
    Return label for a quarter row exactly as in FFADescription.
    Most likely 'Q4 25', but change if needed after looking at the print.
    """
    q = (dt.month - 1) // 3 + 1
    return f"Q{q} {dt.strftime('%y')}"
    # If you see 'Q4-25' use:
    # return f"Q{q}-{dt.strftime('%y')}"
    # If you see 'Q4 2025' use:
    # return f"Q{q} {dt.year}"

# --- build next 12 months from 1 jan 2026 ---
start = datetime(2026, 2, 1)
months = pd.date_range(start, periods=12, freq="MS")

rows = []
for dt in months:
    mon_lab = month_label(dt)
    q_lab = quarter_label(dt)

    # Monthly first
    row_m = latest_df[latest_df["FFADescription"] == mon_lab]

    if not row_m.empty:
        use_row = row_m.iloc[0]
        source = "Monthly"
    else:
        # Quarterly fallback
        row_q = latest_df[latest_df["FFADescription"] == q_lab]
        if not row_q.empty:
            use_row = row_q.iloc[0]
            source = "Quarterly"
        else:
            use_row = None
            source = None

    rows.append({
        "Month": dt.strftime("%Y-%m"),
        "FFA_Description_Used": None if use_row is None else use_row["FFADescription"],
        "HS7TC_FFA": None if use_row is None else use_row["RouteAverage"],
        "Source": source
    })

# ... all your existing code that builds result_df ...

result_df = pd.DataFrame(rows)
print(result_df)

# save to CSV (no index column)
output_path = r"C:\Users\NehaMathew\Documents\Myproject\FFA NEXT 12 MONTHS\Handysize_FFA_12m_strip.csv"
result_df.to_csv(output_path, index=False)



Unique FFADescription for latest 5TC rows:
['Aug 26', 'Cal 27', 'Cal 28', 'Cal 29', 'Cal 30', 'Cal 31', 'Cal 32', 'Cal 33', 'Dec 26', 'Jul 26', 'Jun 26', 'Nov 26', 'Oct 26', 'Q1 27', 'Q2 26', 'Q2 27', 'Q3 26', 'Q3 27', 'Q4 26', 'Q4 27', 'Sep 26']

      RouteIdentifier  RouteAverage FFADescription
28948     TC_H38+2MON       16365.0         Aug 26
28960     TC_H38+1CAL       12725.0         Cal 27
28961     TC_H38+2CAL       11800.0         Cal 28
28962     TC_H38+3CAL       11560.0         Cal 29
28963     TC_H38+4CAL       11405.0         Cal 30
28964     TC_H38+5CAL       11305.0         Cal 31
28965     TC_H38+6CAL       11350.0         Cal 32
28966     TC_H38+7CAL       11300.0         Cal 33
28952     TC_H38+6MON       15080.0         Dec 26
28947     TC_H38+1MON       16910.0         Jul 26
28946    TC_H38CURMON       16040.0         Jun 26
28951     TC_H38+5MON       15360.0         Nov 26
28950     TC_H38+4MON       15540.0         Oct 26
28956       TC_H38+3Q       11910.0   

In [4]:
import pandas as pd
from datetime import datetime

# Paths
ffa_path = r"C:\Users\NehaMathew\Documents\Myproject\FFA NEXT 12 MONTHS\Handysize_FFA_12m_strip.csv"
spot_path = r"C:\Users\NehaMathew\Documents\Myproject\Daily spot Ratio\Spot_Ratio_Data_Week 24 2026.xlsx"
current_date_str = datetime.today().strftime("%Y-%m-%d")
baltic_path = fr"C:\Users\NehaMathew\Documents\Baltic\Baltic_Data_average_{current_date_str}.xlsx"

# Original Handysize ratio columns
RATIO_COLS_OLD = [
    
    "HS1_38 VS HS7TC",
    "HS2_38 VS HS7TC",
    "HS3_38 VS HS7TC",
    "HS4_38 VS HS7TC",
    "HS5_38 VS HS7TC",
    "HS6_38 VS HS7TC",
    "HS7_38 VS HS7TC"
]

# New names with '_ratio' suffix
RATIO_COLS_NEW = [col + "_ratio" for col in RATIO_COLS_OLD]

def build_block_for_years(year_list, label):
    ffa_df = pd.read_csv(ffa_path)
    ffa_df["Month_dt"] = pd.to_datetime(ffa_df["Month"] + "-01")
    ffa_df["MonthNum"] = ffa_df["Month_dt"].dt.month

    spot_df = pd.read_excel(spot_path, sheet_name="Handysize ratio data")
    spot_df["Date"] = pd.to_datetime(spot_df["Date"], errors="coerce")
    spot_df["Year"] = spot_df["Date"].dt.year
    spot_df["MonthNum"] = spot_df["Date"].dt.month

    spot_sel = spot_df[spot_df["Year"].isin(year_list)].copy()

    # Compute monthly averages
    monthly_avgs = (
        spot_sel.groupby("MonthNum", as_index=False)[RATIO_COLS_OLD].mean()
    )
    monthly_avgs[RATIO_COLS_OLD] = monthly_avgs[RATIO_COLS_OLD].round(2)

    # Rename to *_ratio
    monthly_avgs.rename(columns=dict(zip(RATIO_COLS_OLD, RATIO_COLS_NEW)), inplace=True)

    merged = ffa_df.merge(monthly_avgs, how="left", on="MonthNum")

    # Estimated FFA basis (HS7TC_FFA is the base)
    for col in range(1, 8):
        base = f"HS{col}_38"
        merged[f"{base}_estimated_FFA basis"] = merged["HS7TC_FFA"] * merged[f"{base} VS HS7TC_ratio"]

    # Placeholder spot basis
    for col in range(1, 8):
        base = f"HS{col}_38"
        merged[f"{base}_estimated_Spot basis"] = 0

    # Round FFA estimates
    est_ffa_cols = [f"HS{i}_38_estimated_FFA basis" for i in range(1, 8)]
    merged[est_ffa_cols] = merged[est_ffa_cols].round(0)

    merged = merged.drop(columns=["Month_dt", "MonthNum"])
    merged["Year"] = label

    # Final column list
    cols = ["Month", "FFA_Description_Used", "HS7TC_FFA", "Source"]

    for col in range(1, 8):
        base = f"HS{col}_38"
        cols += [
            f"{base} VS HS7TC_ratio",
            f"{base}_estimated_FFA basis",
            f"{base}_estimated_Spot basis"
        ]

    cols.append("Year")

    return merged[cols]

# Read latest Baltic spot price
baltic_df = pd.read_excel(baltic_path, sheet_name="Daily")
baltic_df["Date"] = pd.to_datetime(baltic_df["Date"], errors="coerce")
latest_date = baltic_df["Date"].max()
latest_hs7tc_spot = baltic_df.loc[baltic_df["Date"] == latest_date, "HS7TC"].values[0]
date_str = latest_date.strftime("%d_%m_%Y")
spot_col_name = f"HS7TC_{date_str}_Spot"

# Build year blocks 2018–2025
year_blocks = [build_block_for_years([y], y) for y in range(2018, 2027)]

# Relative year blocks
base_year = 2025
last2_block = build_block_for_years([base_year - 1, base_year], "Last 2 years")
last3_block = build_block_for_years([base_year - 2, base_year - 1, base_year], "Last 3 years")
last4_block = build_block_for_years([base_year - 3, base_year - 2, base_year - 1, base_year], "Last 4 years")
last5_block = build_block_for_years([base_year - 4, base_year - 3, base_year - 2, base_year - 1, base_year], "Last 5 years")

# Combine everything
full_df = pd.concat(year_blocks + [last2_block, last3_block, last4_block, last5_block], ignore_index=True)

# Add spot column
full_df[spot_col_name] = latest_hs7tc_spot
hs7_idx = full_df.columns.get_loc("HS7TC_FFA")
full_df = full_df.drop(columns=[spot_col_name])
full_df.insert(hs7_idx + 1, spot_col_name, latest_hs7tc_spot)

# Calculate estimated Spot basis
for col in range(1, 8):
    base = f"HS{col}_38"
    ratio_col = f"{base} VS HS7TC_ratio"
    est_spot_col = f"{base}_estimated_Spot basis"
    full_df[est_spot_col] = (full_df[spot_col_name] * full_df[ratio_col]).round(0)

# Save output
output_path = r"C:\Users\NehaMathew\Downloads\Handysize_FFA_12m_full_with_last_ratios_and_spot.csv"
full_df.to_csv(output_path, index=False)

print(full_df.head(40))


      Month FFA_Description_Used  HS7TC_FFA  HS7TC_12_06_2026_Spot     Source  \
0   2026-02                  NaN        NaN                16195.0        NaN   
1   2026-03                  NaN        NaN                16195.0        NaN   
2   2026-04                Q2 26    14856.0                16195.0  Quarterly   
3   2026-05                Q2 26    14856.0                16195.0  Quarterly   
4   2026-06               Jun 26    16040.0                16195.0    Monthly   
5   2026-07               Jul 26    16910.0                16195.0    Monthly   
6   2026-08               Aug 26    16365.0                16195.0    Monthly   
7   2026-09               Sep 26    15890.0                16195.0    Monthly   
8   2026-10               Oct 26    15540.0                16195.0    Monthly   
9   2026-11               Nov 26    15360.0                16195.0    Monthly   
10  2026-12               Dec 26    15080.0                16195.0    Monthly   
11  2027-01                Q

In [5]:
import pandas as pd
from datetime import datetime

path = r"C:\Users\NehaMathew\Documents\FFA\input\Supramax_FFA.csv"

df = pd.read_csv(path, sep=None, engine="python")

# Parse ArchiveDate
df["ArchiveDate"] = pd.to_datetime(df["ArchiveDate"], errors="coerce")

# Latest 5TC snapshot
latest = df["ArchiveDate"].max()
latest_df = df[
    (df["ArchiveDate"] == latest) &
    (df["RouteIdentifier"].astype(str).str.startswith("10TC_S"))
].copy()

# Clean description a bit
latest_df["FFADescription"] = latest_df["FFADescription"].astype(str).str.strip()

print("Unique FFADescription for latest 5TC rows:")
print(sorted(latest_df["FFADescription"].unique()))
print()

# Inspect one example monthly and one quarterly row so we know formats
print(latest_df[["RouteIdentifier", "RouteAverage", "FFADescription"]]
      .sort_values("FFADescription")
      .head(20))

# --- helper functions that you may need to tweak after seeing the above print ---

def month_label(dt: datetime) -> str:
    # Your data: 'Dec 25', 'Jan 26', ...  (space instead of dash)
    return dt.strftime("%b %y")   # e.g. 'Dec 25'


def quarter_label(dt: datetime) -> str:
    """
    Return label for a quarter row exactly as in FFADescription.
    Most likely 'Q4 25', but change if needed after looking at the print.
    """
    q = (dt.month - 1) // 3 + 1
    return f"Q{q} {dt.strftime('%y')}"
    # If you see 'Q4-25' use:
    # return f"Q{q}-{dt.strftime('%y')}"
    # If you see 'Q4 2025' use:
    # return f"Q{q} {dt.year}"

# --- build next 12 months from 1 jan 2026 ---
start = datetime(2026, 2, 1)
months = pd.date_range(start, periods=12, freq="MS")

rows = []
for dt in months:
    mon_lab = month_label(dt)
    q_lab = quarter_label(dt)

    # Monthly first
    row_m = latest_df[latest_df["FFADescription"] == mon_lab]

    if not row_m.empty:
        use_row = row_m.iloc[0]
        source = "Monthly"
    else:
        # Quarterly fallback
        row_q = latest_df[latest_df["FFADescription"] == q_lab]
        if not row_q.empty:
            use_row = row_q.iloc[0]
            source = "Quarterly"
        else:
            use_row = None
            source = None

    rows.append({
        "Month": dt.strftime("%Y-%m"),
        "FFA_Description_Used": None if use_row is None else use_row["FFADescription"],
        "S10TC_FFA": None if use_row is None else use_row["RouteAverage"],
        "Source": source
    })

# ... all your existing code that builds result_df ...

result_df = pd.DataFrame(rows)
print(result_df)

# save to CSV (no index column)
output_path = r"C:\Users\NehaMathew\Documents\Myproject\FFA NEXT 12 MONTHS\Supramax_FFA_12m_strip.csv"
result_df.to_csv(output_path, index=False)



Unique FFADescription for latest 5TC rows:
['Aug 26', 'Cal 27', 'Cal 28', 'Cal 29', 'Cal 30', 'Cal 31', 'Cal 32', 'Cal 33', 'Dec 26', 'Jul 26', 'Jun 26', 'Nov 26', 'Oct 26', 'Q1 27', 'Q2 26', 'Q2 27', 'Q3 26', 'Q3 27', 'Q4 26', 'Q4 27', 'Sep 26']

      RouteIdentifier  RouteAverage FFADescription
53603     10TC_S+2MON       19450.0         Aug 26
53615     10TC_S+1CAL       13741.0         Cal 27
53616     10TC_S+2CAL       12546.0         Cal 28
53617     10TC_S+3CAL       12100.0         Cal 29
53618     10TC_S+4CAL       11893.0         Cal 30
53619     10TC_S+5CAL       11889.0         Cal 31
53620     10TC_S+6CAL       11804.0         Cal 32
53621     10TC_S+7CAL       11757.0         Cal 33
53607     10TC_S+6MON       17375.0         Dec 26
53602     10TC_S+1MON       19786.0         Jul 26
53601    10TC_SCURMON       18857.0         Jun 26
53606     10TC_S+5MON       17832.0         Nov 26
53605     10TC_S+4MON       18296.0         Oct 26
53611       10TC_S+3Q       13093.0   

In [6]:
import pandas as pd
from datetime import datetime

# Paths
ffa_path = r"C:\Users\NehaMathew\Documents\Myproject\FFA NEXT 12 MONTHS\Supramax_FFA_12m_strip.csv"
spot_path = r"C:\Users\NehaMathew\Documents\Myproject\Daily spot Ratio\Spot_Ratio_Data_Week 24 2026.xlsx"
current_date_str = datetime.today().strftime("%Y-%m-%d")
baltic_path = fr"C:\Users\NehaMathew\Documents\Baltic\Baltic_Data_average_{current_date_str}.xlsx"

# Supramax ratio routes (excluding S10TC which is the base)
RATIO_BASES = [
    "S1B_58",
    "S1C_58",
    "S2_58",
    "S3_58",
    "S4A_58",
    "S4B_58",
    "S5_58",
    "S8_58",
    "S9_58"
]

# Original ratio column names in SPOT data
RATIO_COLS_OLD = [f"{col} VS S10TC" for col in RATIO_BASES]

# New names with '_ratio' suffix
RATIO_COLS_NEW = [f"{col}_ratio" for col in RATIO_COLS_OLD]

def build_block_for_years(year_list, label):
    ffa_df = pd.read_csv(ffa_path)
    ffa_df["Month_dt"] = pd.to_datetime(ffa_df["Month"] + "-01")
    ffa_df["MonthNum"] = ffa_df["Month_dt"].dt.month

    spot_df = pd.read_excel(spot_path, sheet_name="Supramax ratio data")
    spot_df["Date"] = pd.to_datetime(spot_df["Date"], errors="coerce")
    spot_df["Year"] = spot_df["Date"].dt.year
    spot_df["MonthNum"] = spot_df["Date"].dt.month

    spot_sel = spot_df[spot_df["Year"].isin(year_list)].copy()

    # Compute monthly averages
    monthly_avgs = (
        spot_sel.groupby("MonthNum", as_index=False)[RATIO_COLS_OLD].mean()
    )
    monthly_avgs[RATIO_COLS_OLD] = monthly_avgs[RATIO_COLS_OLD].round(2)

    # Rename for merging
    monthly_avgs.rename(columns=dict(zip(RATIO_COLS_OLD, RATIO_COLS_NEW)), inplace=True)

    merged = ffa_df.merge(monthly_avgs, how="left", on="MonthNum")

    # Estimated FFA basis (using S10TC_FFA)
    for base in RATIO_BASES:
        merged[f"{base}_estimated_FFA basis"] = (
            merged["S10TC_FFA"] * merged[f"{base} VS S10TC_ratio"]
        )

    # Placeholder spot basis columns
    for base in RATIO_BASES:
        merged[f"{base}_estimated_Spot basis"] = 0

    # Round FFA estimates
    est_ffa_cols = [f"{base}_estimated_FFA basis" for base in RATIO_BASES]
    merged[est_ffa_cols] = merged[est_ffa_cols].round(0)

    merged = merged.drop(columns=["Month_dt", "MonthNum"])
    merged["Year"] = label

    # Column order
    cols = ["Month", "FFA_Description_Used", "S10TC_FFA", "Source"]
    for base in RATIO_BASES:
        cols += [
            f"{base} VS S10TC_ratio",
            f"{base}_estimated_FFA basis",
            f"{base}_estimated_Spot basis"
        ]
    cols.append("Year")

    return merged[cols]

# Read latest Baltic spot price
baltic_df = pd.read_excel(baltic_path, sheet_name="Daily")
baltic_df["Date"] = pd.to_datetime(baltic_df["Date"], errors="coerce")
latest_date = baltic_df["Date"].max()
latest_s10tc_spot = baltic_df.loc[baltic_df["Date"] == latest_date, "S10TC"].values[0]

date_str = latest_date.strftime("%d_%m_%Y")
spot_col_name = f"S10TC_{date_str}_Spot"

# Build year blocks 2018–2025
year_blocks = [build_block_for_years([y], y) for y in range(2018, 2027)]

# Last N-year blocks
base_year = 2025
last2_block = build_block_for_years([base_year - 1, base_year], "Last 2 years")
last3_block = build_block_for_years([base_year - 2, base_year - 1, base_year], "Last 3 years")
last4_block = build_block_for_years([base_year - 3, base_year - 2, base_year - 1, base_year], "Last 4 years")
last5_block = build_block_for_years([base_year - 4, base_year - 3, base_year - 2, base_year - 1, base_year], "Last 5 years")

# Stack together
full_df = pd.concat(
    year_blocks + [last2_block, last3_block, last4_block, last5_block],
    ignore_index=True
)

# Insert S10TC spot column
full_df[spot_col_name] = latest_s10tc_spot
idx = full_df.columns.get_loc("S10TC_FFA")
full_df = full_df.drop(columns=[spot_col_name])
full_df.insert(idx + 1, spot_col_name, latest_s10tc_spot)

# Compute Spot basis for each route
for base in RATIO_BASES:
    full_df[f"{base}_estimated_Spot basis"] = (
        full_df[spot_col_name] * full_df[f"{base} VS S10TC_ratio"]
    ).round(0)

# Save output
output_path = r"C:\Users\NehaMathew\Downloads\Supramax_FFA_12m_full_with_last_ratios_and_spot.csv"
full_df.to_csv(output_path, index=False)

print(full_df.head(40))


      Month FFA_Description_Used  S10TC_FFA  S10TC_12_06_2026_Spot     Source  \
0   2026-02                  NaN        NaN                18722.0        NaN   
1   2026-03                  NaN        NaN                18722.0        NaN   
2   2026-04                Q2 26    17338.0                18722.0  Quarterly   
3   2026-05                Q2 26    17338.0                18722.0  Quarterly   
4   2026-06               Jun 26    18857.0                18722.0    Monthly   
5   2026-07               Jul 26    19786.0                18722.0    Monthly   
6   2026-08               Aug 26    19450.0                18722.0    Monthly   
7   2026-09               Sep 26    19100.0                18722.0    Monthly   
8   2026-10               Oct 26    18296.0                18722.0    Monthly   
9   2026-11               Nov 26    17832.0                18722.0    Monthly   
10  2026-12               Dec 26    17375.0                18722.0    Monthly   
11  2027-01                Q

In [7]:
import pandas as pd
from datetime import datetime

path = r"C:\Users\NehaMathew\Documents\FFA\input\Panamax_FFA.csv"

df = pd.read_csv(path, sep=None, engine="python")

# Parse ArchiveDate
df["ArchiveDate"] = pd.to_datetime(df["ArchiveDate"], errors="coerce")

# Latest 5TC snapshot
latest = df["ArchiveDate"].max()
latest_df = df[
    (df["ArchiveDate"] == latest) &
    (df["RouteIdentifier"].astype(str).str.startswith("5TC_P"))
].copy()

# Clean description a bit
latest_df["FFADescription"] = latest_df["FFADescription"].astype(str).str.strip()

print("Unique FFADescription for latest 5TC rows:")
print(sorted(latest_df["FFADescription"].unique()))
print()

# Inspect one example monthly and one quarterly row so we know formats
print(latest_df[["RouteIdentifier", "RouteAverage", "FFADescription"]]
      .sort_values("FFADescription")
      .head(20))

# --- helper functions that you may need to tweak after seeing the above print ---

def month_label(dt: datetime) -> str:
    # Your data: 'Dec 25', 'Jan 26', ...  (space instead of dash)
    return dt.strftime("%b %y")   # e.g. 'Dec 25'


def quarter_label(dt: datetime) -> str:
    """
    Return label for a quarter row exactly as in FFADescription.
    Most likely 'Q4 25', but change if needed after looking at the print.
    """
    q = (dt.month - 1) // 3 + 1
    return f"Q{q} {dt.strftime('%y')}"
    # If you see 'Q4-25' use:
    # return f"Q{q}-{dt.strftime('%y')}"
    # If you see 'Q4 2025' use:
    # return f"Q{q} {dt.year}"

# --- build next 12 months from 1 Dec 2025 ---
start = datetime(2026, 2, 1)
months = pd.date_range(start, periods=12, freq="MS")

rows = []
for dt in months:
    mon_lab = month_label(dt)
    q_lab = quarter_label(dt)

    # Monthly first
    row_m = latest_df[latest_df["FFADescription"] == mon_lab]

    if not row_m.empty:
        use_row = row_m.iloc[0]
        source = "Monthly"
    else:
        # Quarterly fallback
        row_q = latest_df[latest_df["FFADescription"] == q_lab]
        if not row_q.empty:
            use_row = row_q.iloc[0]
            source = "Quarterly"
        else:
            use_row = None
            source = None

    rows.append({
        "Month": dt.strftime("%Y-%m"),
        "FFA_Description_Used": None if use_row is None else use_row["FFADescription"],
        "P5TC_FFA": None if use_row is None else use_row["RouteAverage"],
        "Source": source
    })

# ... all your existing code that builds result_df ...

result_df = pd.DataFrame(rows)
print(result_df)

# save to CSV (no index column)
output_path = r"C:\Users\NehaMathew\Documents\Myproject\FFA NEXT 12 MONTHS\Panamax_FFA_12m_strip.csv"
result_df.to_csv(output_path, index=False)



Unique FFADescription for latest 5TC rows:
['Aug 26', 'Cal 27', 'Cal 28', 'Cal 29', 'Cal 30', 'Cal 31', 'Cal 32', 'Cal 33', 'Dec 26', 'Jul 26', 'Jun 26', 'Nov 26', 'Oct 26', 'Q1 27', 'Q2 26', 'Q2 27', 'Q3 26', 'Q3 27', 'Q4 26', 'Q4 27', 'Sep 26']

      RouteIdentifier  RouteAverage FFADescription
51840      5TC_P+2MON       21043.0         Aug 26
51852      5TC_P+1CAL       15702.0         Cal 27
51853      5TC_P+2CAL       14471.0         Cal 28
51854      5TC_P+3CAL       13832.0         Cal 29
51855      5TC_P+4CAL       13579.0         Cal 30
51856      5TC_P+5CAL       13307.0         Cal 31
51857      5TC_P+6CAL       13250.0         Cal 32
51858      5TC_P+7CAL       13179.0         Cal 33
51844      5TC_P+6MON       19146.0         Dec 26
51839      5TC_P+1MON       21479.0         Jul 26
51838     5TC_PCURMON       20807.0         Jun 26
51843      5TC_P+5MON       19761.0         Nov 26
51842      5TC_P+4MON       19961.0         Oct 26
51848        5TC_P+3Q       15354.0   

In [8]:
import pandas as pd
from datetime import datetime

# Paths
ffa_path = r"C:\Users\NehaMathew\Documents\Myproject\FFA NEXT 12 MONTHS\Panamax_FFA_12m_strip.csv"
spot_path = r"C:\Users\NehaMathew\Documents\Myproject\Daily spot Ratio\Spot_Ratio_Data_Week 24 2026.xlsx"

current_date_str = datetime.today().strftime("%Y-%m-%d")
baltic_path = fr"C:\Users\NehaMathew\Documents\Baltic\Baltic_Data_average_{current_date_str}.xlsx"

# Panamax ratio columns (old names)
RATIO_COLS_OLD = [
    "P1A_82 VS P5TC",
    "P2A_82 VS P5TC",
    "P3A_82 VS P5TC",
    "P4_82 VS P5TC",
    "P6_82 VS P5TC"
]

# New ratio names with "_ratio"
RATIO_COLS_NEW = [col + "_ratio" for col in RATIO_COLS_OLD]


def build_block_for_years(year_list, label):

    # --- Load FFA Strip ---
    ffa_df = pd.read_csv(ffa_path)
    ffa_df["Month_dt"] = pd.to_datetime(ffa_df["Month"] + "-01")
    ffa_df["MonthNum"] = ffa_df["Month_dt"].dt.month

    # --- Load Spot Ratio Data ---
    spot_df = pd.read_excel(spot_path, sheet_name="Panamax ratio data")
    spot_df["Date"] = pd.to_datetime(spot_df["Date"], errors="coerce")
    spot_df["Year"] = spot_df["Date"].dt.year
    spot_df["MonthNum"] = spot_df["Date"].dt.month

    spot_sel = spot_df[spot_df["Year"].isin(year_list)].copy()

    # Monthly averages of ratios
    monthly_avgs = (
        spot_sel.groupby("MonthNum", as_index=False)[RATIO_COLS_OLD].mean()
    )
    monthly_avgs[RATIO_COLS_OLD] = monthly_avgs[RATIO_COLS_OLD].round(2)

    # Rename to new names
    monthly_avgs.rename(columns=dict(zip(RATIO_COLS_OLD, RATIO_COLS_NEW)), inplace=True)

    # Merge with FFA strip
    merged = ffa_df.merge(monthly_avgs, how="left", on="MonthNum")

    # --- Estimated FFA basis ---
    merged["P1A_estimated_FFA basis"] = merged["P5TC_FFA"] * merged["P1A_82 VS P5TC_ratio"]
    merged["P2A_estimated_FFA basis"] = merged["P5TC_FFA"] * merged["P2A_82 VS P5TC_ratio"]
    merged["P3A_estimated_FFA basis"] = merged["P5TC_FFA"] * merged["P3A_82 VS P5TC_ratio"]
    merged["P4_estimated_FFA basis"]  = merged["P5TC_FFA"] * merged["P4_82 VS P5TC_ratio"]
    merged["P6_estimated_FFA basis"]  = merged["P5TC_FFA"] * merged["P6_82 VS P5TC_ratio"]

    # Placeholder for spot basis
    for col in ["P1A", "P2A", "P3A", "P4", "P6"]:
        merged[f"{col}_estimated_Spot basis"] = 0

    # Round estimated FFA basis
    ffa_cols = [
        "P1A_estimated_FFA basis", "P2A_estimated_FFA basis",
        "P3A_estimated_FFA basis", "P4_estimated_FFA basis",
        "P6_estimated_FFA basis"
    ]
    merged[ffa_cols] = merged[ffa_cols].round(0)

    merged = merged.drop(columns=["Month_dt", "MonthNum"])
    merged["Year"] = label

    # Final column order
    cols = [
        "Month", "FFA_Description_Used", "P5TC_FFA", "Source",
        "P1A_82 VS P5TC_ratio", "P1A_estimated_FFA basis", "P1A_estimated_Spot basis",
        "P2A_82 VS P5TC_ratio", "P2A_estimated_FFA basis", "P2A_estimated_Spot basis",
        "P3A_82 VS P5TC_ratio", "P3A_estimated_FFA basis", "P3A_estimated_Spot basis",
        "P4_82 VS P5TC_ratio", "P4_estimated_FFA basis", "P4_estimated_Spot basis",
        "P6_82 VS P5TC_ratio", "P6_estimated_FFA basis", "P6_estimated_Spot basis",
        "Year"
    ]

    return merged[cols]


# --- Load Latest Baltic Spot ---
baltic_df = pd.read_excel(baltic_path, sheet_name="Daily")
baltic_df["Date"] = pd.to_datetime(baltic_df["Date"], errors="coerce")
latest_date = baltic_df["Date"].max()
latest_p5tc_spot = baltic_df.loc[baltic_df["Date"] == latest_date, "P5TC"].values[0]

date_str = latest_date.strftime("%d_%m_%Y")
spot_col_name = f"P5TC_{date_str}_Spot"


# --- Build Year Blocks for 2018–2025 ---
year_blocks = [build_block_for_years([y], y) for y in range(2018, 2027)]

# --- Last N years ---
base_year = 2025
last2 = build_block_for_years([base_year - 1, base_year], "Last 2 years")
last3 = build_block_for_years([base_year - 2, base_year - 1, base_year], "Last 3 years")
last4 = build_block_for_years([base_year - 3, base_year - 2, base_year - 1, base_year], "Last 4 years")
last5 = build_block_for_years([base_year - 4, base_year - 3, base_year - 2, base_year - 1, base_year], "Last 5 years")

# Stack all
full_df = pd.concat(year_blocks + [last2, last3, last4, last5], ignore_index=True)

# Add P5TC spot column
full_df[spot_col_name] = latest_p5tc_spot
p5tc_idx = full_df.columns.get_loc("P5TC_FFA")
full_df = full_df.drop(columns=[spot_col_name])
full_df.insert(p5tc_idx + 1, spot_col_name, latest_p5tc_spot)

# --- Calculate Estimated Spot Basis ---
for col in ["P1A", "P2A", "P3A", "P4", "P6"]:
    ratio_col = f"{col}_82 VS P5TC_ratio"
    est_col = f"{col}_estimated_Spot basis"
    full_df[est_col] = (full_df[spot_col_name] * full_df[ratio_col]).round(0)

# Save output
output_path = r"C:\Users\NehaMathew\Downloads\Panamax_FFA_12m_full_with_last_ratios_and_spot.csv"
full_df.to_csv(output_path, index=False)

print(full_df.head(40))


      Month FFA_Description_Used  P5TC_FFA  P5TC_12_06_2026_Spot     Source  \
0   2026-02                  NaN       NaN               20545.0        NaN   
1   2026-03                  NaN       NaN               20545.0        NaN   
2   2026-04                Q2 26   19592.0               20545.0  Quarterly   
3   2026-05                Q2 26   19592.0               20545.0  Quarterly   
4   2026-06               Jun 26   20807.0               20545.0    Monthly   
5   2026-07               Jul 26   21479.0               20545.0    Monthly   
6   2026-08               Aug 26   21043.0               20545.0    Monthly   
7   2026-09               Sep 26   20454.0               20545.0    Monthly   
8   2026-10               Oct 26   19961.0               20545.0    Monthly   
9   2026-11               Nov 26   19761.0               20545.0    Monthly   
10  2026-12               Dec 26   19146.0               20545.0    Monthly   
11  2027-01                Q1 27   15354.0          

In [9]:
import pandas as pd

# File paths for your CSV files
csv_files = [
    r"C:\Users\NehaMathew\Downloads\Panamax_FFA_12m_full_with_last_ratios_and_spot.csv",
    r"C:\Users\NehaMathew\Downloads\Supramax_FFA_12m_full_with_last_ratios_and_spot.csv",
    r"C:\Users\NehaMathew\Downloads\Handysize_FFA_12m_full_with_last_ratios_and_spot.csv",
    r"C:\Users\NehaMathew\Downloads\Capesize_FFA_12m_full_with_last_ratios_and_spot.csv"
]

# Output Excel file name
output_excel = r"C:\Users\NehaMathew\Downloads\combined.xlsx"

# Create ExcelWriter to write multiple sheets
with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    for i, csv_file in enumerate(csv_files, start=1):
        # Read each CSV
        df = pd.read_csv(csv_file)
        
        # Write to a separate sheet
        sheet_name = f"Sheet{i}"   # or use a custom name
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Excel file created successfully!")


Excel file created successfully!
